In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# ============================================
# 0. 工具函数：Hilbert 射影度量 & 精度
# ============================================

def hilbert_distance(w: torch.Tensor, v: torch.Tensor, eps: float = 1e-12) -> float:
    """
    在正锥内部向量 w, v > 0 上的 Hilbert 投影度量:
        d_H(w, v) = log( max_i (w_i / v_i) / min_i (w_i / v_i) )
    w, v: shape (D,)
    返回 python float
    """
    w = torch.clamp(w, min=eps)
    v = torch.clamp(v, min=eps)
    ratio = w / v
    r_min = torch.min(ratio)
    r_max = torch.max(ratio)
    d = torch.log(r_max / r_min)
    return float(d.item())


def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    """
    logits: (n,)
    y:      (n,) int64, 0/1
    """
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).long()
    acc = (preds == y).float().mean()
    return float(acc.item())


# ============================================
# 1. 生成 Swiss roll + 高维嵌入 + 线性可分标签
# ============================================

def generate_swiss_roll_binary_linear_sep(
    n_samples: int,
    D: int = 32,
    noise: float = 0.0,
    device: str = "cpu",
):
    """
    生成 Swiss roll 数据并嵌入到 R^D:
      - 参数 (u, v)
      - 3D: (x1, x2, x3)
      - 高维嵌入: X_high = A @ x_3d, A ∈ R^{D x 3}, 列正交

    二分类标签：
      先在 span(A) 中选一个 w_true，再用线性函数
        score_i = <w_true, X_i> + b_true
      定义
        y_i = 1_{score_i > 0}
      这样在 X_high 空间里是**严格线性可分**的。

    返回:
      X_high: (n_samples, D)
      y:      (n_samples,)  -- 0/1
      u, v:   (n_samples,)  -- 方便你以后做内在几何分析
      A:      (D, 3)        -- 嵌入矩阵（列正交）
      w_true: (D,)          -- 真实超平面法向量（不一定全正）
      b_true: ()            -- 真实 bias
    """
    # 内在参数
    u = torch.empty(n_samples, device=device).uniform_(3 * math.pi, 9 * math.pi)
    v = torch.empty(n_samples, device=device).uniform_(0.0, 20.0)

    # 3D swiss roll
    x = torch.zeros(n_samples, 3, device=device)
    x[:, 0] = u * torch.cos(u)
    x[:, 1] = v
    x[:, 2] = u * torch.sin(u)

    if noise > 0.0:
        x = x + noise * torch.randn_like(x)

    # 构造高维嵌入矩阵 A: D x 3, 列正交
    A_random = torch.randn(D, 3, device=device)
    Q, _ = torch.linalg.qr(A_random)  # Q: D x 3
    A = Q

    # 高维嵌入
    X_high = x @ A.T  # (n,3) @ (3,D) = (n,D)

    # 在 span(A) 中选一个 w_true：
    # 先随一个 3 维向量 c，再映射 w_true = A c
    c = torch.tensor([1.0, 0.5, -0.3], device=device)  # 你也可以改成别的方向
    c = c / torch.norm(c)
    w_true = A @ c  # (D,)

    # 线性打分 + 设置 bias = - median，保证两类大致均衡
    scores = X_high @ w_true  # (n,)
    b_true = -scores.median()
    scores_shifted = scores + b_true

    # 线性可分标签
    y = (scores_shifted > 0).long()

    return X_high, y, u, v, A, w_true, b_true


# ============================================
# 2. 模型：正锥参数化线性 Logistic 回归
# ============================================

class PositiveLinearLogistic(nn.Module):
    """
    w = exp(z) ∈ R_{>0}^D, b ∈ R.
    logits(x) = <w, x> + b.
    """

    def __init__(self, D: int):
        super().__init__()
        # z 是 unconstrained 参数
        self.z = nn.Parameter(torch.zeros(D))
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        """
        x: (n, D)
        返回:
          logits: (n,)
          w:      (D,)  当前正锥参数
        """
        w = torch.exp(self.z)  # 强制正
        logits = x @ w + self.b  # (n,)
        return logits, w


# ============================================
# 3. 主实验函数
# ============================================

def run_experiment(
    n_samples: int = 4000,
    D: int = 32,
    device: str = None,
    batch_size: int = None,
    ref_epochs: int = 800,
    ref_lr: float = 1e-2,
    n_runs: int = 3,
    T: int = 200,
    lr: float = 5e-2,
):
    """
    整个实验管线：
      1）生成数据（瑞士卷二分类，线性可分）
      2）训练一个“参考解” w*
      3）多次随机初始化，记录 GD 的动力学量
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. 数据生成（线性可分）
    X, y, u, v, A, w_true, b_true = generate_swiss_roll_binary_linear_sep(
        n_samples=n_samples,
        D=D,
        noise=0.1,
        device=device,
    )

    # 训练 / 测试切分
    n_train = int(0.8 * n_samples)
    perm = torch.randperm(n_samples, device=device)
    train_idx = perm[:n_train]
    test_idx = perm[n_train:]

    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    if batch_size is None:
        batch_size = n_train  # 全量 GD，方便分析

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
    )

    # 2. 构造 projection 到 span(A)
    # A: (D,3), 列正交 => 投影矩阵 P = A A^T
    def project_to_spanA(w_vec: torch.Tensor) -> torch.Tensor:
        """
        w_vec: (D,)
        返回 w_parallel = P_S w，S = span(A)
        """
        coeff = A.T @ w_vec      # (3,)
        w_parallel = A @ coeff   # (D,)
        return w_parallel

    print("\nTrue hyperplane summary:")
    print("  ||w_true||_2 =", float(torch.norm(w_true).item()))
    print("  b_true      =", float(b_true.item()))

    # 3. 先训练参考解 w*
    print("\n=== Training reference solution w* ===")
    model_ref = PositiveLinearLogistic(D).to(device)
    optimizer_ref = torch.optim.SGD(model_ref.parameters(), lr=ref_lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(ref_epochs):
        model_ref.train()
        for xb, yb in train_loader:
            optimizer_ref.zero_grad()
            logits, w = model_ref(xb)
            loss = criterion(logits, yb.float())
            loss.backward()
            optimizer_ref.step()

        if (epoch + 1) % max(1, ref_epochs // 10) == 0:
            with torch.no_grad():
                logits_train, w_cur = model_ref(X_train)
                train_loss = criterion(logits_train, y_train.float()).item()
                train_acc = accuracy_from_logits(logits_train, y_train)
                logits_test, _ = model_ref(X_test)
                test_acc = accuracy_from_logits(logits_test, y_test)
            print(
                f"[Ref Epoch {epoch+1:4d}/{ref_epochs}] "
                f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                f"test_acc={test_acc:.4f}"
            )

    with torch.no_grad():
        _, w_star = model_ref(X_train)
        w_star = w_star.detach()  # (D,)

    print("\nReference w* summary:")
    print("  ||w*||_2 =", float(torch.norm(w_star).item()))

    # 4. 多次随机初始化，记录动力学
    print("\n=== Running dynamics from multiple initializations ===")
    all_runs_stats = []

    for run in range(n_runs):
        print(f"\n--- Run {run+1}/{n_runs} ---")
        model = PositiveLinearLogistic(D).to(device)
        # 重新随机初始化 z, b
        with torch.no_grad():
            model.z.normal_(mean=0.0, std=0.1)
            model.b.zero_()

        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

        # 存储轨迹
        traj_loss = []
        traj_acc = []
        traj_dH_to_star = []
        traj_dH_to_init = []
        traj_norm_para = []
        traj_norm_perp = []

        with torch.no_grad():
            _, w0 = model(X_train)
            w0 = w0.detach()
            w0_parallel = project_to_spanA(w0)
            w0_perp = w0 - w0_parallel

        dH_0_star = hilbert_distance(w0, w_star)
        dH_0_0 = hilbert_distance(w0, w0)  # 0

        print(f"  Initial d_H(w0, w*): {dH_0_star:.6f}")
        print(f"  Initial d_H(w0, w0): {dH_0_0:.6f}")
        print(
            f"  Initial ||w0_parallel||={float(torch.norm(w0_parallel)):.4f}, "
            f"||w0_perp||={float(torch.norm(w0_perp)):.4f}"
        )

        for t in range(T):
            model.train()
            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits, w = model(xb)
                loss = criterion(logits, yb.float())
                loss.backward()
                optimizer.step()

            with torch.no_grad():
                logits_all, w_t = model(X_train)
                loss_t = criterion(logits_all, y_train.float()).item()
                acc_t = accuracy_from_logits(logits_all, y_train)

                # Hilbert 距离
                dH_star = hilbert_distance(w_t, w_star)
                dH_init = hilbert_distance(w_t, w0)

                # 分解到 span(A) 和其正交补
                w_t_parallel = project_to_spanA(w_t)
                w_t_perp = w_t - w_t_parallel
                norm_para = float(torch.norm(w_t_parallel).item())
                norm_perp = float(torch.norm(w_t_perp).item())

            traj_loss.append(loss_t)
            traj_acc.append(acc_t)
            traj_dH_to_star.append(dH_star)
            traj_dH_to_init.append(dH_init)
            traj_norm_para.append(norm_para)
            traj_norm_perp.append(norm_perp)

            if (t + 1) % max(1, T // 10) == 0 or t == 0:
                print(
                    f"  [t={t+1:4d}/{T}] "
                    f"loss={loss_t:.4f}, acc={acc_t:.4f}, "
                    f"dH(w_t, w*)={dH_star:.6f}, dH(w_t, w0)={dH_init:.6f}, "
                    f"||para||={norm_para:.4f}, ||perp||={norm_perp:.4f}"
                )

        run_stats = {
            "loss": traj_loss,
            "acc": traj_acc,
            "dH_to_star": traj_dH_to_star,
            "dH_to_init": traj_dH_to_init,
            "norm_para": traj_norm_para,
            "norm_perp": traj_norm_perp,
        }
        all_runs_stats.append(run_stats)

    print("\n=== Experiment finished. ===")
    print("你可以把 all_runs_stats 存到文件或后面用 matplotlib 画图分析。")

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "u": u,
        "v": v,
        "A": A,
        "w_true": w_true,
        "b_true": b_true,
        "w_star": w_star,
        "runs": all_runs_stats,
    }


if __name__ == "__main__":
    results = run_experiment(
        n_samples=4000,
        D=32,
        ref_epochs=800,
        ref_lr=1e-2,
        n_runs=3,
        T=200,
        lr=5e-2,
    )
    # 你可以在这里添加代码把 results 存到文件，或者画图分析

Using device: cuda

True hyperplane summary:
  ||w_true||_2 = 0.9999999403953552
  b_true      = -4.2303900718688965

=== Training reference solution w* ===
[Ref Epoch   80/800] train_loss=0.0999, train_acc=0.9503, test_acc=0.9500
[Ref Epoch  160/800] train_loss=0.0950, train_acc=0.9541, test_acc=0.9525
[Ref Epoch  240/800] train_loss=0.0944, train_acc=0.9537, test_acc=0.9525
[Ref Epoch  320/800] train_loss=0.0937, train_acc=0.9537, test_acc=0.9525
[Ref Epoch  400/800] train_loss=0.0931, train_acc=0.9547, test_acc=0.9525
[Ref Epoch  480/800] train_loss=0.0926, train_acc=0.9553, test_acc=0.9525
[Ref Epoch  560/800] train_loss=0.0920, train_acc=0.9553, test_acc=0.9525
[Ref Epoch  640/800] train_loss=0.0915, train_acc=0.9553, test_acc=0.9525
[Ref Epoch  720/800] train_loss=0.0909, train_acc=0.9553, test_acc=0.9525
[Ref Epoch  800/800] train_loss=0.0904, train_acc=0.9559, test_acc=0.9537

Reference w* summary:
  ||w*||_2 = 6.364463806152344

=== Running dynamics from multiple initializatio